> Benjamin Nikholas

> Data Science / JCSDOL-014

> Modul 3 - Tugas 17

> 22 Juli 2024
---
---

In [97]:
import pandas as pd
pd.set_option('display.max_columns', 30)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Target : LeaveOrNot 

In [34]:
df = pd.read_csv('Employee.csv')
df.info()
df.head(5)

X = df.drop(columns = 'LeaveOrNot')
y = df['LeaveOrNot']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Education                  4653 non-null   object
 1   JoiningYear                4653 non-null   int64 
 2   City                       4653 non-null   object
 3   PaymentTier                4653 non-null   int64 
 4   Age                        4653 non-null   int64 
 5   Gender                     4653 non-null   object
 6   EverBenched                4653 non-null   object
 7   ExperienceInCurrentDomain  4653 non-null   int64 
 8   LeaveOrNot                 4653 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 327.3+ KB


Lakukan Feature Selection

In [35]:
# Check categorical features unique count
# before encode
X_categ_cols = X.select_dtypes('object').columns

for i in X_categ_cols:
    print(X[i].value_counts(),'\n')

Education
Bachelors    3601
Masters       873
PHD           179
Name: count, dtype: int64 

City
Bangalore    2228
Pune         1268
New Delhi    1157
Name: count, dtype: int64 

Gender
Male      2778
Female    1875
Name: count, dtype: int64 

EverBenched
No     4175
Yes     478
Name: count, dtype: int64 



In [36]:
# Encode categorical features for chi-squared test

label_encoders = {}
for cols in X_categ_cols:
    le = LabelEncoder()
    X[cols] = le.fit_transform(df[cols])
    label_encoders[cols] = le

In [37]:
# Check categorical features unique count
# after encode

for i in X_categ_cols:
    print(X[i].value_counts(),'\n')

Education
0    3601
1     873
2     179
Name: count, dtype: int64 

City
0    2228
2    1268
1    1157
Name: count, dtype: int64 

Gender
1    2778
0    1875
Name: count, dtype: int64 

EverBenched
0    4175
1     478
Name: count, dtype: int64 



In [38]:
chi2_selector = SelectKBest(score_func = chi2, 
                            k = 'all')
chi2_selector.fit(X, y)
chi2_scores = chi2_selector.scores_

# Get feature names and scores
features = X.columns
feature_scores = pd.DataFrame({'Feature': features, 
                               'Score': chi2_scores}).sort_values(
                                   by = 'Score', 
                                   ascending = False
                               ).reset_index(drop = True)
                               
display(feature_scores)

,Feature,Score
0,City,167.972324
1,Gender,91.328840
2,Education,30.942168
3,EverBenched,25.686589
4,PaymentTier,21.227341
5,Age,9.635261
6,ExperienceInCurrentDomain,3.617279
7,JoiningYear,0.264660


In [73]:
# Select features

feature_select = feature_scores.iloc[:5]['Feature'].values
X_feat_select = df[feature_select]
X_feat_select.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   City         4653 non-null   object
 1   Gender       4653 non-null   object
 2   Education    4653 non-null   object
 3   EverBenched  4653 non-null   object
 4   PaymentTier  4653 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 181.9+ KB


Lakukan Feature Engineering 

In [92]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), X_feat_select.select_dtypes(include = 'int64').columns),
        ('cat', OneHotEncoder(), X_feat_select.select_dtypes(exclude = 'int64').columns)
    ])

X_processed = preprocessor.fit_transform(X_feat_select)

X_processed = pd.DataFrame(X_processed, 
                           columns = preprocessor.get_feature_names_out())

In [93]:
display(X_processed)

,num__PaymentTier,cat__City_Bangalore,cat__City_New Delhi,cat__City_Pune,cat__Gender_Female,cat__Gender_Male,cat__Education_Bachelors,cat__Education_Masters,cat__Education_PHD,cat__EverBenched_No,cat__EverBenched_Yes
0,0.537503,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
1,-3.025177,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.537503,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.537503,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.537503,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
4648,0.537503,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4649,-1.243837,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4650,0.537503,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4651,0.537503,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0


Split data menjadi Train 80% dan Test 20% 

In [95]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y,
                                                    train_size = 0.8,
                                                    random_state = 1)

Lakukan Modelling menggunakan Logistic Regression 

In [104]:
model = LogisticRegression(random_state = 1, 
                           max_iter = 100)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

Accuracy: 0.71
Confusion Matrix:
 [[525  61]
 [211 134]]
Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.90      0.79       586
           1       0.69      0.39      0.50       345

    accuracy                           0.71       931
   macro avg       0.70      0.64      0.65       931
weighted avg       0.70      0.71      0.68       931



Cek Multicollinearity 

In [113]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['Feature'] = X_processed.columns
vif_data['VIF'] = [variance_inflation_factor(X_processed.values, i) for i in range(X_processed.shape[1])]
vif_data['VIF'] = vif_data['VIF'].round(0).astype('int')
vif_data = vif_data.sort_values(by = 'VIF',
                                ascending = False).reset_index(drop = True)

display(vif_data)

,Feature,VIF
0,cat__Education_PHD,1340251926
1,cat__Education_Masters,45744
2,cat__City_New Delhi,17672
3,cat__Gender_Female,9239
4,cat__City_Bangalore,6576
5,cat__EverBenched_Yes,5934
6,cat__City_Pune,3509
7,cat__Gender_Male,505
8,cat__Education_Bachelors,348
9,cat__EverBenched_No,181


Buat Interpretasi Hasil Summary 

In [114]:
accuracy = accuracy_score(y_test, y_pred).round(2)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print('Accuracy:', accuracy)
print('Confusion Matrix:\n', conf_matrix)
print('Classification Report:\n', class_report)

Accuracy: 0.71
Confusion Matrix:
 [[525  61]
 [211 134]]
Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.90      0.79       586
           1       0.69      0.39      0.50       345

    accuracy                           0.71       931
   macro avg       0.70      0.64      0.65       931
weighted avg       0.70      0.71      0.68       931



Tentukan Features mana yg paling penting 

In [128]:
coefficients = model.coef_[0]

# Create a DataFrame to display feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': abs(coefficients)
})

feature_importance = feature_importance.sort_values(by = 'Coefficient', 
                                                    ascending = False).reset_index(drop = True)

display(feature_importance)

,Feature,Coefficient
0,cat__Education_Masters,0.628623
1,cat__City_Pune,0.611351
2,cat__City_New Delhi,0.570308
3,cat__Gender_Male,0.471936
4,cat__Gender_Female,0.471684
5,cat__Education_Bachelors,0.329865
6,cat__EverBenched_No,0.313722
7,cat__EverBenched_Yes,0.313469
8,cat__Education_PHD,0.299010
9,num__PaymentTier,0.212986


In [143]:
best_feature, best_val = feature_importance.iloc[0].values

print(f'Fitur yang paling penting adalah fitur "{best_feature}" dengan nilai koefisien absolut {best_val.round(3)}')

Fitur yang paling penting adalah fitur "cat__Education_Masters" dengan nilai koefisien absolut 0.629


: 